# Chi-Squared Tests

# Moving from Numeric to Categorical Data

## What We've Covered So Far:
- **Statistical hypothesis testing framework**
- **T-tests** for investigating differences between numeric variables
- Point estimates and confidence intervals

## Next Topic: Categorical Variables
- Now we'll focus on **categorical variables** (not numeric)
- Different type of data requires different statistical tests

## The Chi-Squared Test:
- A common statistical test for **categorical variables**
- Used when your data consists of categories or groups
- Different from t-tests which work with numeric data

## Examples of Categorical Data:
- Gender (Male, Female, Other)
- Political party (Democrat, Republican, Independent)
- Product preference (Brand A, Brand B, Brand C)
- Survey responses (Yes, No, Maybe)

# Chi-Squared Goodness-Of-Fit Test

## Comparison to T-Tests:
- **One-sample t-test**: Checks if sample mean differs from expected (population) mean
- **Chi-squared goodness-of-fit test**: Checks if sample categorical distribution matches expected distribution

## What It Tests:
- Whether the **distribution** of sample categorical data matches an expected distribution
- Not the individual values, but how they're distributed across categories

## Real-World Examples:
- **School demographics**: Do member demographics match U.S. population demographics?
- **Browser preferences**: Do your friends' browser choices match general Internet users?
- **Product preferences**: Does your sample match expected market preferences?

## Why Categorical Data is Different:
- Categories like "male", "female", "other" have **no mathematical meaning**
- You can't do math with category names
- Instead, we work with **counts** of how many fall into each category

## Example Setup:
- Generate fake demographic data for **U.S. population** (expected distribution)
- Generate fake demographic data for **Minnesota sample**
- Test: Does Minnesota's demographic distribution match the U.S. distribution?

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [2]:
# Create data frames for national and Minnesota populations

# Distribution of race in the US
national = pd.DataFrame(["white"]*100000 + ["hispanic"]*60000 +\
                        ["black"]*50000 + ["asian"]*15000 + ["other"]*35000)

# Distribution of race in Minnesota
minnesota = pd.DataFrame(["white"]*600 + ["hispanic"]*300 + \
                         ["black"]*250 +["asian"]*75 + ["other"]*150) 

# Create frequency tables
national_table = pd.crosstab(index=national[0], columns="count")
minnesota_table = pd.crosstab(index=minnesota[0], columns="count")

print( "National")
print(national_table)
print(" ")
print( "Minnesota")
print(minnesota_table)

National
col_0      count
0               
asian      15000
black      50000
hispanic   60000
other      35000
white     100000
 
Minnesota
col_0     count
0              
asian        75
black       250
hispanic    300
other       150
white       600


## Chi-Squared Statistic Formula:

### The Formula:
χ² = Σ [(observed - expected)² / expected]


### Breaking Down the Formula:
- **Observed**: The actual count you see in each category in your sample
- **Expected**: The count you would expect based on the population distribution
- **Σ (sigma)**: Sum this calculation across all categories

### How It Works:
1. For each category, subtract expected from observed
2. Square that difference: `(observed - expected)²`
3. Divide by the expected count: `/ expected`
4. Add up all these values across all categories

### What the Statistic Tells Us:
- **Large χ² value**: Big difference between observed and expected → reject null hypothesis
- **Small χ² value**: Small difference between observed and expected → fail to reject null hypothesis

### Example Calculation:
Let's calculate the chi-squared statistic for our U.S. vs Minnesota demographic data step by step:

In [3]:

observed = minnesota_table

national_ratios = national_table/len(national)  # Get population ratios

print(national_ratios)

expected = national_ratios * len(minnesota)   # Get expected counts

# Calculate Chi-squared statistic
chi_squared_stat = (((observed-expected)**2)/expected).sum()

print(f"Result: {chi_squared_stat}")

col_0        count
0                 
asian     0.057692
black     0.192308
hispanic  0.230769
other     0.134615
white     0.384615
Result: col_0
count    18.194805
dtype: float64


## Important Assumption:
- **Note**: The chi-squared test assumes **none of the expected counts are less than 5**
- If any expected count < 5, the test may not be reliable

## Comparing to Critical Values:

### What We'll Do:
1. Find the **critical value** for 95% confidence level
2. Compare our calculated χ² statistic to this critical value
3. Check the **p-value** of our result

### Decision Rule:
- If χ² statistic > critical value → reject null hypothesis
- If χ² statistic ≤ critical value → fail to reject null hypothesis

In [4]:
crit = stats.chi2.ppf(q = 0.95, # Find the critical value for 95% confidence*
                      df = 4)   # Df = number of variable categories - 1

print("Critical value")
print(crit)

# Find the p-value
p_value = 1 - stats.chi2.cdf(x=chi_squared_stat,  
                             df=4)
print("P value")
print(p_value)

Critical value
9.487729036781154
P value
[0.00113047]


Since our chi-squared statistic exceeds the critical value, we'd reject the null hypothesis that the two distributions are the same.

You can carry out a chi-squared goodness-of-fit test automatically using the scipy function scipy.stats.chisquare():

In [5]:
stats.chisquare(f_obs= observed,   # Array of observed counts
                f_exp= expected)   # Array of expected counts

Power_divergenceResult(statistic=array([18.19480519]), pvalue=array([0.00113047]))

In [6]:
0.05 > 0.00113047

True

The test results agree with the values we calculated above.

# Chi-Squared Test of Independence

## Independence in Probability

### What is Independence?
- **Independence**: When knowing the value of one variable tells you **nothing** about another variable
- If variables are independent, they don't influence each other

### Examples of Independent Variables:
- **Birth month** and **web browser preference**
  - Knowing someone was born in January doesn't help predict if they use Chrome or Firefox
- **Eye color** and **favorite pizza topping**
  - These traits aren't related to each other

### Examples of Non-Independent Variables:
- **Birth month** and **sports performance in school**
  - Kids born in certain months might have age advantages in youth sports
- **Education level** and **income**
  - These are often related to each other

## Chi-Squared Test of Independence

### Common Applications:
- Do **political views** vary by **gender**?
- Does **education level** differ by **race**?
- Are **product preferences** related to **age group**?
- Do **religious beliefs** vary by **geographic region**?

### Example Setup:
- Generate fake **voter polling data**
- Test whether voting preference is independent of demographic factors
- Use chi-squared test of independence to find the answer

In [7]:
np.random.seed(10)

# Sample data randomly at fixed probabilities
voter_race = np.random.choice(a= ["asian","black","hispanic","other","white"],
                              p = [0.05, 0.15 ,0.25, 0.05, 0.5],
                              size=1000)

# Sample data randomly at fixed probabilities
voter_party = np.random.choice(a= ["democrat","independent","republican"],
                              p = [0.4, 0.2, 0.4],
                              size=1000)

# Create a data frame from the two arrays
voters = pd.DataFrame({"race":voter_race, 
                       "party":voter_party})

# Create a contingency table
voter_tab = pd.crosstab(voters.race, voters.party, margins = True)

# Label the rows and columns
voter_tab.columns = ["democrat","independent","republican","row_totals"]

voter_tab.index = ["asian","black","hispanic","other","white","col_totals"]

observed = voter_tab.iloc[0:5,0:3]   # Get table without totals for later use
voter_tab

,democrat,independent,republican,row_totals
asian,21,7,32,60
black,65,25,64,154
hispanic,107,50,94,251
other,15,8,15,38
white,189,96,212,497
col_totals,397,186,417,1000


In [8]:
# Calculate expected values
# expected value = (row total * column total) / grand total
expected =  np.outer(voter_tab["row_totals"][0:5],
                     voter_tab.loc["col_totals"][0:3]) / 1000

# Convert to data frame for easier handling
expected = pd.DataFrame(expected)

# Label the rows and columns
expected.columns = ["democrat","independent","republican"]
expected.index = ["asian","black","hispanic","other","white"]

expected

,democrat,independent,republican
asian,23.820,11.160,25.020
black,61.138,28.644,64.218
hispanic,99.647,46.686,104.667
other,15.086,7.068,15.846
white,197.309,92.442,207.249


In [9]:
# Calculate Chi-squared statistic
stats.chisquare(f_obs= observed,   # Array of observed counts
                f_exp= expected)   # Array of expected counts

Power_divergenceResult(statistic=array([1.47078796, 2.50934232, 3.189191  ]), pvalue=array([0.83180311, 0.64296369, 0.52667855]))

## Test Results:

### Output Breakdown:
- **Test statistics (χ²)**: [1.471, 2.509, 3.189]
- **P-values**: [0.832, 0.643, 0.527]
- Multiple values because we're testing multiple relationships

### Interpreting the P-Values:
- **First test**: p-value = 0.832 (83.2%)
- **Second test**: p-value = 0.643 (64.3%)  
- **Third test**: p-value = 0.527 (52.7%)

### Decision Making:
- **Significance level**: Usually 5% (α = 0.05)
- **All our p-values** are much higher than 5%
- Since **all p-values > 0.05** → **Fail to reject null hypothesis** for all tests

### What This Means:
- We found **no evidence** that the variables are dependent on each other
- The variables appear to be **independent**
- Any patterns we see could reasonably be due to random chance

### Practical Interpretation:
- The demographic factors we tested don't seem to predict voting preferences
- Knowing someone's demographic information doesn't help predict their vote
- The variables are likely **independent** of each other

# Wrap Up

Chi-squared tests provide a way to investigate differences in the distributions of categorical variables with the same categories and the dependence between categorical variables. In the next lesson, we'll learn about a third statistical inference test, the analysis of variance, that lets us compare several sample means at the same time.

### References:
1. https://www.kaggle.com/code/hamelg/python-for-data-25-chi-squared-tests/notebook
2. Seabold, Skipper, and Josef Perktold. “statsmodels: Econometric and statistical modeling with python.” Proceedings of the 9th Python in Science Conference. 2010.